# Bird Inference Pipeline (YOLO + Species Classifier)
Notebook for development and verification of JPEG bird detection + species identification.


## 1) Environment + Imports


In [1]:
from pathlib import Path
import pandas as pd
from PIL import Image

from bird_pipeline import (
    PipelineConfig,
    InputItem,
    get_default_device,
    list_classifier_checkpoints,
    list_jpeg_files_from_folder,
    load_classifier,
    load_yolo,
    detect_best_bird,
    crop_detection_fullres,
    classify_crop,
    run_inference_batch,
)


## 2) Model Loading Helpers


In [2]:
ckpts = list_classifier_checkpoints("artifacts")
print("Checkpoints:", len(ckpts))
for c in ckpts[:5]:
    print(" -", c)

device = get_default_device()
print("Default device:", device)

classifier_checkpoint = ckpts[0] if ckpts else None
label_csv = "artifacts/label_names.csv"
yolo_weights = "yolo11n.pt"

if classifier_checkpoint:
    clf, labels = load_classifier(classifier_checkpoint, label_csv, device=device)
    yolo = load_yolo(yolo_weights)
    print("Loaded classifier labels:", len(labels))


Checkpoints: 22
 - artifacts/resnet50_nabirds_head_only_decay_1e-4_tt_50-50.pt
 - artifacts/resnet50_nabirds_head_only_decay_2e-4_tt_50-50.pt
 - artifacts/resnet50_nabirds_head_only_decay_2e-4_tt_80-20.pt
 - artifacts/resnet50_nabirds_head_only_decay_2e-4_tt_80-20_lr_4e-3_batch_num_test_64_r01_20260214_230109.pt
 - artifacts/resnet50_nabirds_head_only_decay_2e-4_tt_80-20_lr_8e-3_bs_128_batch_num_test_128_r01_20260214_230109.pt
Default device: mps
Loaded classifier labels: 98


## 3) JPEG Discovery + Filtering Helpers


In [3]:
sample_folder = "NABirds Dataset/nabirds/images"
files = list_jpeg_files_from_folder(sample_folder, recursive=False)
print("Found JPEG files (top-level only):", len(files))
print(files[:3])


Found JPEG files (top-level only): 0
[]


## 4) YOLO Bird-Detection Helper


In [4]:
# Update image_path to any local JPEG for testing
image_path = None
if files:
    # pick first file for a quick smoke test
    image_path = str(files[0])

if image_path and Path(image_path).exists() and classifier_checkpoint:
    img = Image.open(image_path).convert("RGB")
    det = detect_best_bird(img, yolo_model=yolo, conf=0.25, device=device)
    print(det)
else:
    print("Set image_path to a valid JPEG to test detection.")


Set image_path to a valid JPEG to test detection.


## 5) Full-Resolution Crop Helper


In [5]:
if image_path and Path(image_path).exists() and classifier_checkpoint:
    img = Image.open(image_path).convert("RGB")
    det = detect_best_bird(img, yolo_model=yolo, conf=0.25, device=device)
    if det is not None:
        crop = crop_detection_fullres(img, det.bbox_xyxy)
        if crop is not None:
            print("Crop size:", crop.size)
            display(crop)
        else:
            print("Degenerate crop.")
    else:
        print("No bird detected.")


## 6) Species Classification Helper


In [6]:
if image_path and Path(image_path).exists() and classifier_checkpoint:
    img = Image.open(image_path).convert("RGB")
    det = detect_best_bird(img, yolo_model=yolo, conf=0.25, device=device)
    if det is not None:
        crop = crop_detection_fullres(img, det.bbox_xyxy)
        if crop is not None:
            pred = classify_crop(crop, classifier_model=clf, label_names=labels, device=device)
            print("Top-1:", pred["species"], f"({pred['confidence']:.3f})")
            print("Top-5:")
            for r in pred["top5"]:
                print(f"  {r['rank']}. {r['species']} ({r['confidence']:.3f})")


## 7) End-to-End Batch Run (Folder)


In [7]:
# Set this to your own folder of JPEGs
my_folder = ""
recursive = True

if my_folder:
    paths = list_jpeg_files_from_folder(my_folder, recursive=recursive)
    inputs = [
        InputItem(
            source_type="folder",
            source_path=str(p),
            image_name=p.name,
            pil_image=Image.open(p).convert("RGB"),
        )
        for p in paths
    ]

    if not inputs:
        print("No JPEGs found.")
    else:
        cfg = PipelineConfig(
            classifier_checkpoint=classifier_checkpoint,
            label_names_csv=label_csv,
            yolo_weights=yolo_weights,
            yolo_conf=0.25,
            device="auto",
            output_root="artifacts/pipeline_runs",
        )

        results, errors, run_dir = run_inference_batch(inputs, cfg)
        print("Run dir:", run_dir)
        print("Results:", len(results), "Errors:", len(errors))
        display(pd.DataFrame([r.__dict__ for r in results]).head(20))
else:
    print("Set my_folder to run batch inference on your photos.")


Set my_folder to run batch inference on your photos.
